# v26 学習ノート — v9特徴(smooth PF注入 lag512) → LightGBM×3 + CatBoost×3 + XGBoost×3
- 入力: `v26/artifacts_v26/data/train.parquet`(create_train_parquet_v26.py 生成=v9 build_well特徴, run_pf_ancc=smooth PF)。
- **特徴は v9 と同じ**(pf_ancc/pf_z/beam/formation(tvtF/form/dense)/NCC(sc)/GR統計…)。gsf/heurは無し。
- origin トグル(base=v9の位置/GR/formation等 無接頭, pf_=PF由来, beam_=beam)。model params はハードコード。
- GroupKFold(5) by well で OOF → apply_pp(overall RMSE = Kaggle指標)。fold pkl 保存(再開可)。
- **過学習注意**(docs/overfit_policy.md): GBM+PF特徴は過学習しやすい。OOF↔LB を ±0.2 床込みで見る。


In [1]:
# ===== 設定 + parquet 読み込み + origin トグルで特徴選択 =====
import json, joblib, warnings, gc
from collections import Counter
import numpy as np, pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
warnings.filterwarnings("ignore")

ART     = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v26\artifacts_v26")
PARQUET = ART / "data" / "train.parquet"
MODELS  = ART / "models"; MODELS.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
ROW_FRAC = 1.0     # 学習に使う行の割合(1.0=全行)。OOM/時短で下げる
pp_params = {"alpha": 1.0, "tau": 85, "w_pf": 0.09}   # apply_pp(submitと共有)

# ===== origin トグル(接頭辞ごとに特徴をON/OFF) =====
USE_ORIGIN = {"base": True, "pf": True, "beam": True}
PREFIXES = {"pf": "pf_", "beam": "beam_"}
META = {"well", "id", "target"}
def origin_of(c):
    for o, pre in PREFIXES.items():
        if c.startswith(pre): return o
    return "base"

df = pd.read_parquet(PARQUET)
all_feats = [c for c in df.columns if c not in META]
features = [c for c in all_feats if USE_ORIGIN.get(origin_of(c), True)]
print("origin別 全特徴数:", dict(Counter(origin_of(c) for c in all_feats)))
print("USE_ORIGIN:", USE_ORIGIN, "→ 採用", len(features), "個")
ART.joinpath("features.json").write_text(json.dumps(features))
ART.joinpath("pp_params.json").write_text(json.dumps(pp_params))

if ROW_FRAC < 1.0:
    step = max(1, int(round(1.0 / ROW_FRAC))); df = df.iloc[::step].reset_index(drop=True)
y  = df["target"].to_numpy(np.float32)
g  = df["well"].to_numpy()
lk = df["last_known_tvt"].to_numpy(np.float64) if "last_known_tvt" in df.columns else np.zeros(len(df))
mds = df["md_since"].to_numpy(np.float64) if "md_since" in df.columns else np.zeros(len(df))
pf_res = df["pf_ancc"].to_numpy(np.float64) if "pf_ancc" in df.columns else np.zeros(len(df))  # 基準PF残差
X = df[features].astype("float32"); del df; gc.collect()
print("rows", len(X), "| features", len(features), "| wells", pd.unique(g).size)

origin別 全特徴数: {'base': 307, 'beam': 40, 'pf': 164}
USE_ORIGIN: {'base': True, 'pf': True, 'beam': True} → 採用 511 個
rows 3783989 | features 511 | wells 773


In [2]:
# ===== モデル: LightGBM×3 + CatBoost×3 + XGBoost×3(v20系, GPU) =====
# ※ GPU非搭載/非対応なら: lgb device_type='cpu', cb task_type='CPU', xgb device='cpu' に。
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective="regression", verbose=-1,
         n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=0, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=1, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=2, n_jobs=-1, early_stopping_rounds=250),
]
VARIANTS = ([(f"lightgbm-{i+1}", "lgb", p) for i, p in enumerate(lgb_params)] +
            [(f"catboost-{i+1}", "cb", p) for i, p in enumerate(cb_params)] +
            [(f"xgboost-{i+1}", "xgb", p) for i, p in enumerate(xgb_params)])
print("variants:", [v[0] for v in VARIANTS])

variants: ['lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'catboost-1', 'catboost-2', 'catboost-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']


In [3]:
# ===== GroupKFold(5) で学習 → OOF収集 + fold pkl保存(再開可) =====
def fit_one(kind, params, Xtr, ytr, Xva, yva):
    if kind == "lgb":
        m = LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="rmse",
              callbacks=[early_stopping(250), log_evaluation(0)])
    elif kind == "cb":
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=250, use_best_model=True, verbose=0)
    else:
        m = XGBRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

gkf = GroupKFold(n_splits=N_SPLITS); folds = list(gkf.split(X, y, groups=g))
oof = {name: np.zeros(len(y), np.float32) for name, _, _ in VARIANTS}
for name, kind, params in VARIANTS:
    d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    for k, (tr, va) in enumerate(folds):
        fp = d / f"fold{k}.pkl"
        if fp.exists():
            m = joblib.load(fp); tag = "load"
        else:
            m = fit_one(kind, params, X.iloc[tr], y[tr], X.iloc[va], y[va]); joblib.dump(m, fp); tag = "fit"
        oof[name][va] = m.predict(X.iloc[va])
        print(f"{name} fold{k} [{tag}]: rmse={root_mean_squared_error(y[va], oof[name][va]):.4f}", flush=True)
        del m; gc.collect()
    print(f"== {name} OOF rmse={root_mean_squared_error(y, oof[name]):.4f}")

lightgbm-1 fold0 [load]: rmse=7.1440
lightgbm-1 fold1 [load]: rmse=7.1528
lightgbm-1 fold2 [load]: rmse=7.3005
lightgbm-1 fold3 [load]: rmse=7.6462
lightgbm-1 fold4 [load]: rmse=10.0705
== lightgbm-1 OOF rmse=7.9415
lightgbm-2 fold0 [load]: rmse=7.0868


KeyboardInterrupt: 

In [4]:
# ===== アンサンブル OOF + apply_pp で overall RMSE(=Kaggle指標, CV) =====
names = [v[0] for v in VARIANTS]
oof_ens = np.mean([oof[n] for n in names], axis=0)

def apply_pp_res(res):
    d = res * (1 - pp_params["w_pf"]) + pf_res * pp_params["w_pf"]   # GBM残差 と 基準PF残差 を blend
    if pp_params.get("tau"):
        d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(pp_params["tau"])))
    return d * float(pp_params.get("alpha", 1.0))

raw = root_mean_squared_error(y, oof_ens)
ppr = root_mean_squared_error(y, apply_pp_res(oof_ens))
print("=== OOF overall RMSE(全行プール=Kaggle指標) ===")
print(f"  raw(GBMアンサンブルのみ) : {raw:.4f}")
print(f"  + apply_pp              : {ppr:.4f}")
print(f"  USE_ORIGIN = {USE_ORIGIN}")
print("  ※ sg_smooth は per-well なので OOFでは未適用(submitで適用)。CV↔LB は ±0.2 ノイズ床込みで。")

=== OOF overall RMSE(全行プール=Kaggle指標) ===
  raw(GBMアンサンブルのみ) : 7.8334
  + apply_pp              : 1026.9480
  USE_ORIGIN = {'base': True, 'pf': True, 'beam': True}
  ※ sg_smooth は per-well なので OOFでは未適用(submitで適用)。CV↔LB は ±0.2 ノイズ床込みで。
